# PydanticAI multi-agent handoff

an agent gets a question then it either answers directly or hands off a task/question to another more specialized agent.
- idea is to separate concerns and be more scalable as an agent should be expert in its field
- note that for simpler systems maybe it's better not to separate into several agents - this needs to be evaluated

## Coach

- 3 agents 
- CoachAgent
- StudyAgent
- CareerAgent 

agents as tool

In [9]:
from pathlib import Path
from datetime import datetime
Path("./exports").mkdir(exist_ok=True)

datetime.now().strftime("%Y%m%d_%H%M%S")

'20261030_110202'

In [12]:
from dataclasses import dataclass
from typing import Literal
from model import nvidia_model
from pydantic_ai.agent import Agent
from pydantic_ai import RunContext
from datetime import datetime

model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b")


@dataclass
class ClientDeps:
    name: str
    goal: str
    # defaults -> optional
    experience_lvl: Literal["beginner", "intermediate", "advanced"] = "beginner"
    weekly_hours: int = 8
    current_role: str | None = None


# specialist agents
study_agent = Agent(
    model=model,
    deps_type=ClientDeps,
    instructions=(
        "You are a study expert. Give concrete study plans and learning techniques."
    ),
)

career_agent = Agent(
    model=model,
    deps_type=ClientDeps,
    instructions="You are a career advisor. Give practical advice on jobs, resumes, and career paths.",
)

coach_agent = Agent(
    model=model,
    deps_type=ClientDeps,
    instructions=(
        "You are a personal coach. Follow these steps IN ORDER:\n"
        "1. Call `ask_study_agent` for study questions.\n"
        "2. Call `ask_career_agent` for career questions.\n"
        "3. Combine the answers into ONE short, simple response.\n"
        "4. Call `export_report` with that final response as `text`.\n"
        "5. Reply to the user with the final response."
    ),
)


@study_agent.instructions
def study_name(ctx: RunContext[ClientDeps]) -> str:
    return f"Student's name is {ctx.deps.name}. Address them by name"


@career_agent.instructions
def career_name(ctx: RunContext[ClientDeps]) -> str:
    return f"Person you are advising is {ctx.deps.name}. Address them by name"


@coach_agent.instructions
def coach_name(ctx: RunContext[ClientDeps]) -> str:
    return f"You are coaching {ctx.deps.name}. Address them by name"


# delegation tool
@coach_agent.tool
async def ask_study_agent(ctx: RunContext[ClientDeps], question: str) -> str:
    """ask study specialist for advice on learning and studying"""
    result = await study_agent.run(question, deps=ctx.deps, usage=ctx.usage)
    return result.output


@coach_agent.tool
async def ask_career_agent(ctx: RunContext[ClientDeps], question: str) -> str:
    """ask career specialist for advice on what to choose for career"""
    result = await career_agent.run(question, deps=ctx.deps, usage=ctx.usage)
    return result.output


@coach_agent.tool
def export_report(ctx: RunContext[ClientDeps], text: str):
    """exports the text to specified path"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exports_dir = Path("exports")
    exports_dir.mkdir(exist_ok=True)
    path = exports_dir / f"{ctx.deps.name.casefold()}_{timestamp}.md"
    path.write_text(text)


result = await coach_agent.run(
    "I'm learning Python and want to become a data engineer. What should I do?",
    deps=ClientDeps(
        name="Martin", goal="Looking for studying opportunities or work opportunities"
    ),
)

result

AgentRunResult(output='Martin, here’s your combined roadmap:\n\n**Study plan (learning Python):** Spend ~40% of your time on Python, then move to Pandas, SQLAlchemy/APIs, PySpark, Docker, Airflow/dbt, and cloud basics. Build one small project per week, avoid tutorial hell, and finish with an end-to-end pipeline portfolio project on GitHub (extract → load → transform → orchestrate → test → document).\n\n**Career path (becoming a data engineer):** Data engineers build reliable systems that move and transform data at scale. Master SQL first (it’s daily work), then Python, Git, Linux, one cloud provider (AWS/Azure/GCP), a data warehouse, dbt, Airflow, and optionally Spark/Docker. Build 3 portfolio projects (batch ETL, medallion data lakehouse, real-time streaming), target junior DE/analytics engineer roles, and prepare for SQL/Python/system-design interviews.\n\n**Bottom line:** Start today—install Python, set up a virtual environment, and write a script that reads a file and prints a summ